# EarthScape Climate Agency — Notebook 02: Data Cleaning

### Objective:
Implement a multi-step data cleaning pipeline to handle duplicates, missing values, timestamp validation, coordinate boundaries, invalid durations, and generate a Before vs After cleaning report.


## 1. Import Cleaning Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


## 2. Load Raw Ingested Sample


In [ ]:
df = pd.read_csv('../WeatherEvents_Jan2016-Dec2022.csv', nrows=150000)
initial_count = len(df)
print(f"Initial raw record count: {initial_count:,}")


## 3. Duplicate Detection & Removal


In [ ]:
duplicates_count = df.duplicated().sum()
print(f"Duplicate records found: {duplicates_count:,}")
df.drop_duplicates(inplace=True)


## 4. Missing Value Analysis & Percentage


In [ ]:
missing_summary = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
display(missing_summary[missing_summary['Missing Count'] > 0])


## 5. Timestamp Conversion & Invalid Dates Handling


In [ ]:
df['StartTime(UTC)'] = pd.to_datetime(df['StartTime(UTC)'], errors='coerce')
df['EndTime(UTC)'] = pd.to_datetime(df['EndTime(UTC)'], errors='coerce')
invalid_dates = df['StartTime(UTC)'].isnull().sum() + df['EndTime(UTC)'].isnull().sum()
print(f"Invalid timestamp records dropped: {invalid_dates:,}")
df.dropna(subset=['StartTime(UTC)', 'EndTime(UTC)'], inplace=True)


## 6. Geographic Coordinate Boundary Validation


In [ ]:
# Valid US bounding coordinates (Lat: 18 to 72, Lng: -170 to -60)
valid_coords = (df['LocationLat'] >= 18.0) & (df['LocationLat'] <= 72.0) & (df['LocationLng'] >= -170.0) & (df['LocationLng'] <= -60.0)
print(f"Out-of-bounds coordinate records removed: {(~valid_coords).sum():,}")
df = df[valid_coords]


## 7. Numerical & Categorical Imputation


In [ ]:
# Impute precipitation with 0.0 or median for non-null
df['Precipitation(in)'] = df['Precipitation(in)'].fillna(0.0).clip(lower=0.0, upper=25.0)

# Categorical columns
df['Type'] = df['Type'].fillna('Unknown')
df['Severity'] = df['Severity'].fillna('Moderate')
df['State'] = df['State'].fillna('Unknown')
df['City'] = df['City'].fillna('Unknown')


## 8. Before vs After Cleaning Comparison Report


In [ ]:
final_count = len(df)
report = pd.DataFrame({
    "Metric": ["Original Records", "Duplicates Removed", "Invalid Records Dropped", "Final Cleaned Records", "Data Retention Rate (%)"],
    "Value": [f"{initial_count:,}", f"{duplicates_count:,}", f"{(initial_count - final_count - duplicates_count):,}", f"{final_count:,}", f"{(final_count/initial_count)*100:.2f}%"]
})
display(report)


### Conclusion:
The dataset is fully cleaned, free of invalid coordinates, null timestamps, and ready for feature engineering.
